# Test API Key

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()  # picks up env vars

resp = client.chat.completions.create(
    model="vertex_ai/gemini-2.5-flash",
    messages=[{"role": "user", "content": "Hello"}],
    max_tokens=500,
)
print(resp.choices[0].message.content)

Hello! How can I help you today?


## Agent comparison models

One cell per model, so each connection result is visible on its own.

In [2]:
# Connection test for the five agent-comparison models, all through the gateway.
# Same controls as the harness: max_tokens capped, temperature 0 + seed where the
# API accepts them (the Claude 5 family rejects temperature, so it is not sent).
def probe(model, **controls):
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say hello in one short English sentence."}],
            max_tokens=60,
            **controls,
        )
        choice = resp.choices[0]
        print(f"OK    {model}")
        print(f"      served model : {resp.model}")
        print(f"      finish reason: {choice.finish_reason}")
        print(f"      reply        : {choice.message.content!r}")
    except Exception as exc:
        print(f"FAIL  {model}")
        print(f"      {type(exc).__name__}: {str(exc)[:400]}")


In [3]:
import json
from openai import APIStatusError

try:
    client.chat.completions.create(
        model="gpt-5.6-terra",
        messages=[{"role": "user", "content": "Say hello."}],
        max_tokens=20,
    )
except APIStatusError as exc:
    print("HTTP status:", exc.status_code)
    print(json.dumps(exc.response.json(), indent=2))


### GPT-5.6 Terra (OpenAI route)

In [4]:
probe("gpt-5.6-terra", temperature=1, seed=42)

OK    gpt-5.6-terra
      served model : gpt-5.6-terra
      finish reason: stop
      reply        : 'Hello!'


### GPT-5.6 Sol (OpenAI route)

In [5]:
probe("gpt-5.6-sol", temperature=1, seed=42)

OK    gpt-5.6-sol
      served model : gpt-5.6-sol
      finish reason: stop
      reply        : 'Hello!'


### Claude Sonnet 5 (Anthropic route; temperature is rejected for this model, so not sent)

In [6]:
probe("claude-sonnet-5")

OK    claude-sonnet-5
      served model : claude-sonnet-5
      finish reason: stop
      reply        : 'Hello! How are you doing today?'


### Claude Fable 5 (Anthropic route; temperature is rejected for this model, so not sent)

In [7]:
probe("claude-fable-5")

OK    claude-fable-5
      served model : claude-fable-5
      finish reason: stop
      reply        : "Hello! It's nice to meet you."


### DeepSeek V3.2 (Vertex AI MaaS route)

In [8]:
probe("vertex_ai/deepseek-ai/deepseek-v3.2-maas", temperature=0.0, seed=42)

OK    vertex_ai/deepseek-ai/deepseek-v3.2-maas
      served model : vertex_ai/deepseek-ai/deepseek-v3.2-maas
      finish reason: stop
      reply        : 'Hello.'
